# Segmentasi gigi per gigi — SAM 2

Satu tujuan: **tiap gigi jadi satu mask terpisah.** Tidak ada penomoran FDI, tidak ada metrik.

```
foto → region gigi → satu titik per gigi → SAM 2 → mask per gigi
```

Kuncinya di **titik**. SAM 2 sangat andal kalau diberi satu titik di dalam gigi; yang sulit cuma
menemukan titiknya. Karena itu automatic mask generation ditinggalkan — ia menebar grid titik lalu
berharap, dan pada foto intraoral hasilnya kacau.

```bash
pip install -U "transformers[torch]" scikit-image pillow-heif
```

In [ ]:
CFG = {
    "img_dir":  "Front Teeth drg Laura",
    "out_dir":  "hasil_segmentasi",
    "max_side": 1024,

    # region gigi
    "col_tiles":   12,     # pita kolom untuk ambang lokal (sudut mulut lebih gelap)
    "local_floor": 0.70,

    # titik per gigi
    "smooth_frac":  0.0021,# sigma penghalusan sebelum hitung tepi (x lebar gambar).
                           # Jendela sempit: terlalu kecil -> noise memecah gigi;
                           # terlalu besar -> garis interproksimal ikut kabur.
    "edge_pct":     78,    # persentil tepi -> dipotong, memisahkan gigi yang berimpit
    "min_cut_frac": 0.22,  # buang serpihan setelah pemotongan
    "dedupe_mult":  0.55,  # radius peleburan titik kembar (x distance transform maks)

    # saring mask
    "min_area_frac": 0.0012,
    "max_area_frac": 0.10,
    "dup_iou":       0.60,
}
import os, json
os.makedirs(CFG["out_dir"], exist_ok=True)
print(CFG)

In [ ]:
import glob, time
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
import torch, warnings
warnings.filterwarnings("ignore")
try:
    from pillow_heif import register_heif_opener; register_heif_opener()
except Exception: pass
from scipy import ndimage as ndi
from skimage.filters import threshold_otsu
from skimage.morphology import binary_opening, binary_closing, disk

DEV = torch.device("mps" if torch.backends.mps.is_available()
                   else ("cuda" if torch.cuda.is_available() else "cpu"))

EXTS = ("*.jpg","*.jpeg","*.JPG","*.JPEG","*.png","*.PNG","*.heic","*.HEIC")
paths = sorted(set(sum([glob.glob(os.path.join(CFG["img_dir"], e)) for e in EXTS], [])))
names = [os.path.splitext(os.path.basename(p))[0] for p in paths]

def load_rgb(p):
    im = Image.open(p).convert("RGB")
    if max(im.size) > CFG["max_side"]:
        s = CFG["max_side"]/max(im.size)
        im = im.resize((int(im.width*s), int(im.height*s)), Image.LANCZOS)
    return np.asarray(im, np.uint8)

print(f"{len(paths)} foto | device {DEV}")

## 1. Cari titik — satu per gigi

**Region gigi.** Skor `toothness = terang × (1 − jenuh)`. Gigi terang & pucat, gusi/bibir merah
jenuh. Ambangnya Otsu, dihitung ulang per pita kolom karena sudut mulut jauh lebih gelap.

**Titik per gigi.** Distance transform region → tiap gigi punya "inti", yaitu titik terjauh dari
tepinya. Tapi gigi yang berimpit menyatu jadi satu gumpalan dengan satu inti saja. Karena itu
region dipotong dulu di **garis tepi interproksimal** — batas enamel antar gigi — sehingga tiap
gigi kembali jadi komponen sendiri. Satu titik per komponen.

In [ ]:
def toothness(rgb):
    a = rgb.astype(np.float32)/255.
    mx, mn = a.max(-1), a.min(-1)
    sat = np.where(mx > 1e-6, (mx-mn)/(mx+1e-6), 0.0)
    return mx * (1.0 - sat)

def tooth_region(rgb):
    W = rgb.shape[1]
    t = toothness(rgb)
    g = float(threshold_otsu(t))
    reg = t > g
    edges = np.linspace(0, W, CFG["col_tiles"]+1).astype(int)
    for x0, x1 in zip(edges[:-1], edges[1:]):
        tile = t[:, x0:x1]
        if tile.size < 50 or tile.max() < g: continue
        try: loc = float(threshold_otsu(tile))
        except Exception: continue
        reg[:, x0:x1] = tile > max(min(loc, g), CFG["local_floor"]*g)
    r = max(1, int(0.006*W))
    reg = binary_closing(binary_opening(reg, disk(r)), disk(r))
    lab, n = ndi.label(reg)
    if n:
        sz = ndi.sum(reg, lab, range(1, n+1))
        reg = np.isin(lab, 1 + np.nonzero(sz >= 0.02*sz.max())[0])
    return reg

def find_points(rgb):
    """-> daftar (x, y), satu titik per gigi."""
    reg = tooth_region(rgb)
    if not reg.any(): return [], reg

    # potong di garis tepi -> gigi yang berimpit terpisah kembali.
    # Dihaluskan dulu: tanpa itu noise sensor bikin gradien berbintik dan satu gigi
    # pecah jadi beberapa komponen (pada uji: 9 titik untuk 6 gigi). Garis interproksimal
    # lebarnya beberapa piksel jadi tetap selamat.
    t = ndi.gaussian_filter(toothness(rgb), CFG["smooth_frac"]*rgb.shape[1])
    gy, gx = np.gradient(t)
    edge = np.hypot(gx, gy)
    # "<=" bukan "<": di area berwarna rata gradiennya persis 0, jadi persentilnya juga 0
    # dan perbandingan ketat membuang SEMUA piksel.
    thr = float(np.percentile(edge[reg], CFG["edge_pct"]))
    cut = binary_opening(reg & (edge <= thr), disk(1))
    if not cut.any(): cut = reg.copy()          # jaring pengaman

    lab, n = ndi.label(cut)
    if n == 0: return [], reg
    sz = ndi.sum(cut, lab, range(1, n+1))
    med = float(np.median(sz[sz > 0]))
    pts = []
    for i in range(1, n+1):
        if sz[i-1] < max(CFG["min_cut_frac"]*med, 25): continue
        d = ndi.distance_transform_edt(lab == i)
        y, x = np.unravel_index(int(np.argmax(d)), d.shape)
        pts.append((int(x), int(y)))

    # lebur titik kembar
    dt = ndi.distance_transform_edt(reg)
    rad = max(4, int(CFG["dedupe_mult"]*float(dt.max())))
    keep = []
    for p in pts:
        if all((p[0]-q[0])**2 + (p[1]-q[1])**2 > rad*rad for q in keep): keep.append(p)
    return keep, reg

In [ ]:
# Cek titiknya DULU — ini belum menyentuh SAM, jadi cepat.
# Kalau titiknya sudah satu-per-gigi, sisanya hampir pasti beres.
n_show = min(6, len(paths))
fig, ax = plt.subplots(n_show, 2, figsize=(10, 3.2*n_show))
ax = np.atleast_2d(ax)
for i in range(n_show):
    rgb = load_rgb(paths[i])
    pts, reg = find_points(rgb)
    ax[i,0].imshow(reg, cmap="gray"); ax[i,0].set_title("region gigi", fontsize=8)
    ax[i,1].imshow(rgb)
    if pts:
        x, y = zip(*pts)
        ax[i,1].plot(x, y, "o", ms=7, mfc="lime", mec="black", mew=1)
    ax[i,1].set_title(f"{names[i][:34]} — {len(pts)} titik", fontsize=8)
    for a in ax[i]: a.axis("off")
plt.tight_layout(); plt.show()

## 2. SAM 2 — titik jadi mask

Tiap titik dikirim sebagai prompt positif. `multimask_output=True` memberi 3 kandidat per titik
(biasanya: sebagian gigi / satu gigi utuh / gigi + tetangga) — dipilih yang luasnya wajar dan
paling menempel di region gigi.

In [ ]:
from transformers import Sam2Model, Sam2Processor
model = Sam2Model.from_pretrained("facebook/sam2.1-hiera-large").to(DEV).eval()
processor = Sam2Processor.from_pretrained("facebook/sam2.1-hiera-large")
print("SAM 2 siap")

def _to_masks(out, inp):
    """post_process_masks beda tanda tangan antar versi transformers."""
    pm, osz = out.pred_masks.cpu(), inp["original_sizes"].cpu()
    try:
        return processor.post_process_masks(pm, osz)[0]
    except TypeError:
        return processor.post_process_masks(pm, osz, inp["reshaped_input_sizes"].cpu())[0]

@torch.no_grad()
def sam_masks(rgb, pts):
    if not pts: return []
    inp = processor(images=Image.fromarray(rgb),
                    input_points=[[[list(p)] for p in pts]],
                    input_labels=[[[1] for _ in pts]],
                    return_tensors="pt").to(DEV)
    out = model(**inp, multimask_output=True)
    m = _to_masks(out, inp)                 # (n_titik, n_kandidat, H, W)
    s = out.iou_scores.cpu().numpy()[0]
    return [[(np.asarray(m[i, c], bool), float(s[i, c])) for c in range(m.shape[1])]
            for i in range(m.shape[0])]

def pilih(cands, reg, A):
    """Kandidat yang paling menyerupai SATU gigi."""
    best, bs = None, -9e9
    for mask, sc in cands:
        a = mask.sum()
        if a == 0: continue
        if not (CFG["min_area_frac"] <= a/A <= CFG["max_area_frac"]): continue
        nempel = np.logical_and(mask, reg).sum()/a
        v = 2.0*nempel + sc
        if v > bs: bs, best = v, mask
    return best

def bersihkan(masks, A):
    masks = sorted(masks, key=lambda m: -m.sum())
    keep = []
    for m in masks:
        if any(np.logical_and(m, k).sum()/min(m.sum(), k.sum()) > CFG["dup_iou"] for k in keep):
            continue
        keep.append(m)
    return sorted(keep, key=lambda m: np.nonzero(m)[1].mean())   # kiri -> kanan

def segmentasi(rgb, pts=None):
    if pts is None: pts, reg = find_points(rgb)
    else:           reg = tooth_region(rgb)
    A = rgb.shape[0]*rgb.shape[1]
    hasil = [pilih(c, reg, A) for c in sam_masks(rgb, pts)]
    return bersihkan([m for m in hasil if m is not None], A), pts

In [ ]:
HASIL = {}
t0 = time.time()
for p, n in zip(paths, names):
    rgb = load_rgb(p)
    masks, pts = segmentasi(rgb)
    HASIL[n] = {"rgb": rgb, "masks": masks, "pts": pts}
    print(f"  {n[:44]:44s} {len(pts):2d} titik -> {len(masks):2d} gigi")
print(f"\nselesai dalam {(time.time()-t0)/60:.1f} menit")

## 3. Lihat hasilnya

In [ ]:
def tampil(n, ax=None):
    d = HASIL[n]
    if ax is None: _, ax = plt.subplots(figsize=(7,5))
    ax.imshow(d["rgb"]); ax.axis("off")
    cm = plt.get_cmap("tab20")
    for i, m in enumerate(d["masks"]):
        ax.contour(m, levels=[0.5], colors=[cm(i % 20)], linewidths=1.8)
    ax.set_title(f"{n[:36]} — {len(d['masks'])} gigi", fontsize=8)

ns = list(HASIL); nc = 3; nr = int(np.ceil(len(ns)/nc))
fig, axes = plt.subplots(nr, nc, figsize=(15, 3.7*nr))
for a in np.ravel(axes): a.axis("off")
for k, n in enumerate(ns): tampil(n, np.ravel(axes)[k])
plt.tight_layout(); plt.show()

## 4. Perbaiki yang meleset

Kalau ada gigi yang terlewat atau titiknya salah, tulis titik yang benar di sini dan jalankan
ulang khusus foto itu. Jalankan sel bawah untuk melihat foto bergaris koordinat.

In [ ]:
def grid(n, step=50):
    d = HASIL[n]; rgb = d["rgb"]; H, W = rgb.shape[:2]
    fig, ax = plt.subplots(figsize=(11, 11*H/W))
    ax.imshow(rgb)
    for x in range(0, W, step): ax.axvline(x, color="w", lw=.4, alpha=.6)
    for y in range(0, H, step): ax.axhline(y, color="w", lw=.4, alpha=.6)
    for m in d["masks"]: ax.contour(m, levels=[0.5], colors="cyan", linewidths=.8)
    if d["pts"]:
        x, y = zip(*d["pts"]); ax.plot(x, y, "o", ms=6, mfc="lime", mec="k")
    ax.set_xticks(range(0, W, step*2)); ax.set_yticks(range(0, H, step*2))
    ax.tick_params(labelsize=7); ax.set_title(n[:50], fontsize=9)
    plt.tight_layout(); plt.show()

# grid(names[0])       # <- buka komentarnya untuk membaca koordinat

In [ ]:
# Setel smooth_frac / edge_pct pada FOTO ASLI. Jendela smooth_frac sempit —
# terlalu kecil noise memecah gigi, terlalu besar garis interproksimal ikut kabur.
def sweep(n, param="smooth_frac", nilai=(0.0012, 0.0021, 0.0030, 0.0042)):
    rgb = HASIL[n]["rgb"]; asli = CFG[param]
    fig, ax = plt.subplots(1, len(nilai), figsize=(3.2*len(nilai), 3.6))
    for a, v in zip(np.ravel(ax), nilai):
        CFG[param] = v
        pts, _ = find_points(rgb)
        a.imshow(rgb); a.axis("off")
        if pts:
            x, y = zip(*pts); a.plot(x, y, "o", ms=5, mfc="lime", mec="k", mew=.7)
        a.set_title(f"{param}={v}\n{len(pts)} titik", fontsize=8)
    CFG[param] = asli
    plt.suptitle(f"{n[:46]} — pilih yang satu titik per gigi", fontsize=10)
    plt.tight_layout(); plt.show()

# sweep(names[0])                      # sigma penghalusan
# sweep(names[0], "edge_pct", (68, 74, 78, 84))

In [ ]:
TITIK_MANUAL = {
    # "nama file tanpa ekstensi": [(x1,y1), (x2,y2), ...],
}

for n, pts in TITIK_MANUAL.items():
    if n not in HASIL: print("tidak ada:", n); continue
    masks, _ = segmentasi(HASIL[n]["rgb"], pts=[tuple(p) for p in pts])
    HASIL[n].update({"masks": masks, "pts": [tuple(p) for p in pts]})
    print(f"  {n[:44]:44s} -> {len(masks)} gigi")
    tampil(n); plt.show()

## 5. Simpan

In [ ]:
np.savez_compressed(os.path.join(CFG["out_dir"], "masks.npz"),
    **{f"{n}__{i}": HASIL[n]["masks"][i] for n in HASIL for i in range(len(HASIL[n]["masks"]))})
with open(os.path.join(CFG["out_dir"], "titik.json"), "w") as f:
    json.dump({n: [list(map(int, p)) for p in HASIL[n]["pts"]] for n in HASIL}, f, indent=1)

for n in HASIL:
    print(f"  {n[:46]:46s} {len(HASIL[n]['masks']):2d} gigi")
print(f"\ntotal {sum(len(HASIL[n]['masks']) for n in HASIL)} mask -> {CFG['out_dir']}/")

## Kalau masih meleset

Setel satu per satu, mulai dari yang paling berdampak:

| Gejala | Setel |
|---|---|
| Region gigi bocor ke gusi | naikkan `local_floor` ke 0.8 |
| Gigi pojok hilang dari region | naikkan `col_tiles` ke 16 |
| Dua gigi jadi satu titik | naikkan `edge_pct` ke 84–88 |
| Satu gigi dapat dua titik | naikkan `dedupe_mult` ke 0.7 |
| Mask menjalar ke gusi | turunkan `max_area_frac` ke 0.07 |

Untuk gigi yang tumpang tindih berat, garis interproksimalnya memang nyaris hilang dan tidak ada
setelan yang menolong — pakai `TITIK_MANUAL`. Untuk 18 foto itu jalan tercepat, bukan kegagalan.